<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_ML_site_to_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULLY DATA-DRIVEN SITE-LEVEL ML MODEL
# AGGREGATE TRAFFIC FIRST
# NO ENGINEERING EQUATIONS
# NO MANUAL POWER PARAMETERS
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD EXCEL FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)
site_db.head(2)

In [ ]:
# ============================================================
# CONVERT DATETIME COLUMNS
# ============================================================

site_power['datetime'] = pd.to_datetime(
    site_power['datetime']
)

traffic_4g['datetime'] = pd.to_datetime(
    traffic_4g['datetime']
)

traffic_5g['datetime'] = pd.to_datetime(
    traffic_5g['datetime']
)
traffic_5g.head(2)

In [ ]:
# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]
site_db.head(2)

In [ ]:
# ============================================================
# CREATE TIME FEATURES
# ============================================================

site_power['hour'] = (
    site_power['datetime'].dt.hour
)

site_power['day_of_week'] = (
    site_power['datetime'].dt.dayofweek
)

lte_traffic.rename(
    columns={
        'traffic_load_mbps': 'total_4g_traffic'
    },
    inplace=True
)

In [ ]:
# ============================================================
# AGGREGATE NR TRAFFIC
# ============================================================

nr_traffic = (

    traffic_5g.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

nr_traffic.rename(
    columns={
        'traffic_load_mbps': 'total_5g_traffic'
    },
    inplace=True
)

In [ ]:
# ============================================================
# LTE CELL COUNTS
# ============================================================

lte_cell_counts = (

    traffic_4g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

lte_cell_counts.rename(
    columns={'Cell_ID': 'lte_cell_count'},
    inplace=True
)

In [ ]:
# ============================================================
# NR CELL COUNTS
# ============================================================

nr_cell_counts = (

    traffic_5g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .reset_index()

)

nr_cell_counts.rename(
    columns={'Cell_ID': 'nr_cell_count'},
    inplace=True
)

In [ ]:
# ============================================================
# MERGE ALL DATASETS
# ============================================================

final_df = site_power.merge(

    lte_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

final_df = final_df.merge(

    nr_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

final_df = final_df.merge(

    lte_cell_counts,
    on='Site_ID',
    how='left'

)

final_df = final_df.merge(

    nr_cell_counts,
    on='Site_ID',
    how='left'

)

final_df = final_df.merge(

    site_db,
    on='Site_ID',
    how='left'

)

In [ ]:

# ============================================================
# FILL NULL VALUES
# ============================================================

final_df.fillna(0, inplace=True)

# ============================================================
# FEATURES
# ============================================================

features = [

    'total_4g_traffic',
    'total_5g_traffic',

    'lte_cell_count',
    'nr_cell_count',

    'hour',
    'day_of_week',
    'trigger_ID',

    'RRU_2G',
    'RRU_3G',
    'RRU_4G',

    'AAU_5G',

    'Boards_4G',
    'Boards_5G',

    'BBU5900',
    'BBU3900',
    'BBU3910'

]

X = final_df[features]

y = final_df['site_power']

In [ ]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,
    random_state=42

)

In [ ]:
# ============================================================
# RANDOM FOREST MODEL
# ============================================================

model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

model.fit(X_train, y_train)


In [ ]:

# ============================================================
# PREDICTIONS
# ============================================================

predictions = model.predict(X_test)

In [ ]:
# ============================================================
# EVALUATION METRICS
# ============================================================

mae = mean_absolute_error(

    y_test,
    predictions

)

rmse = np.sqrt(

    mean_squared_error(

        y_test,
        predictions

    )

)

mape = np.mean(

    np.abs(

        (
            y_test - predictions
        )

        /

        y_test

    )

) * 100

r2 = r2_score(

    y_test,
    predictions

)

In [ ]:
# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('SITE-LEVEL ML PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

In [ ]:
# ============================================================
# FINAL RESULTS TABLE
# ============================================================

results_df = final_df.loc[

    X_test.index,

    [
        'Site_ID',
        'trigger_ID',
        'date',
        'datetime',
        'site_power'
    ]

].copy()

results_df['predicted_site_power'] = (
    predictions
)

results_df['error'] = (

    results_df['site_power'] -
    results_df['predicted_site_power']

)

results_df['error_percentage'] = (

    np.abs(results_df['error'])
    /
    results_df['site_power']

) * 100

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance_df = pd.DataFrame({

    'Feature': features,
    'Importance': model.feature_importances_

})

importance_df = importance_df.sort_values(

    by='Importance',
    ascending=False

)

print('================================')
print('FEATURE IMPORTANCE')
print('================================')

print(importance_df)

In [ ]:
# ============================================================
# EXPORT RESULTS
# ============================================================

results_df.to_excel(

    'Fully_Data_Driven_Site_Level_Predictions.xlsx',
    index=False

)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Fully_Data_Driven_Site_Level_Predictions.xlsx')

In [ ]:
# ============================================================
# SAMPLE RESULTS
# ============================================================

print(results_df.head(20))